# SentinelRUL: Multitask Training

Fine tunes the shared GRU backbone jointly on RUL regression and sensor forecasting, starting from the
forecast-pretrained weights produced by `02_kaggle_forecast_training.ipynb`. Run this on Kaggle with GPU
accelerator. Download `multitask_best.pt` from `/kaggle/working/` when finished.

In [ ]:
!pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import os
import math
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print(f'torch {torch.__version__}  cuda: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using device: {DEVICE}')

## Data and Checkpoint Configuration

Point `DATA_DIR` at the Kaggle dataset input folder for the NASA C-MAPSS data, and `FORECAST_CKPT`
at the `forecast_best.pt` file from Stage 6. Upload it via *Add Data > Upload* as a private dataset
and update the path below to match.

In [ ]:
DATA_DIR       = '/kaggle/input/nasa-cmapss-data'
FORECAST_CKPT  = '/kaggle/input/sentinelrul-forecast-checkpoint/forecast_best.pt'

ALL_COLS = ['engine_id', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

DROP_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
SENSOR_COLS  = [c for c in ALL_COLS if c.startswith('s') and c not in DROP_SENSORS]

WINDOW_SIZE      = 30
FORECAST_HORIZON = 5
RUL_CLIP         = 125
VAL_ENGINES      = 20
SEED             = 42
ALPHA            = 0.5

print(f'{len(SENSOR_COLS)} sensors kept: {SENSOR_COLS}')

## Data Loading, Preprocessing, and Windowing

In [ ]:
def load_fd001(data_dir):
    path = os.path.join(data_dir, 'train_FD001.txt')
    df = pd.read_csv(path, sep=r'\s+', header=None, names=ALL_COLS)
    return df


def add_rul_labels(df, rul_clip=RUL_CLIP):
    max_cycle = df.groupby('engine_id')['cycle'].max().rename('max_cycle')
    df = df.join(max_cycle, on='engine_id')
    df['rul'] = (df['max_cycle'] - df['cycle']).clip(upper=rul_clip)
    df.drop(columns=['max_cycle'], inplace=True)
    return df


def fit_scaler(df):
    mins = df[SENSOR_COLS].min()
    maxs = df[SENSOR_COLS].max()
    return mins, maxs


def normalize(df, mins, maxs):
    df = df.copy()
    denom = (maxs - mins).replace(0, 1)
    df[SENSOR_COLS] = (df[SENSOR_COLS] - mins) / denom
    return df


def prepare(data_dir):
    raw = load_fd001(data_dir)
    df  = add_rul_labels(raw.copy())
    mins, maxs = fit_scaler(df)
    df  = normalize(df, mins, maxs)
    return df, mins, maxs


def split_by_engine(df, val_engines=VAL_ENGINES, seed=SEED):
    rng = np.random.default_rng(seed)
    engines = df['engine_id'].unique()
    rng.shuffle(engines)
    val_ids = set(engines[:val_engines])
    return df[~df['engine_id'].isin(val_ids)], df[df['engine_id'].isin(val_ids)]


def make_windows(df, window_size=WINDOW_SIZE, horizon=FORECAST_HORIZON):
    X, y_rul, y_fore = [], [], []
    for _, group in df.groupby('engine_id'):
        sensors = group[SENSOR_COLS].values
        rul     = group['rul'].values
        n = len(sensors)
        for i in range(n - window_size - horizon + 1):
            X.append(sensors[i : i + window_size])
            y_rul.append(rul[i + window_size - 1])
            y_fore.append(sensors[i + window_size : i + window_size + horizon])
    return (
        np.array(X,      dtype=np.float32),
        np.array(y_rul,  dtype=np.float32),
        np.array(y_fore, dtype=np.float32),
    )


class CMAPSSDataset(Dataset):
    def __init__(self, X, y_rul, y_fore):
        self.X      = torch.from_numpy(X)
        self.y_rul  = torch.from_numpy(y_rul)
        self.y_fore = torch.from_numpy(y_fore)

    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y_rul[i], self.y_fore[i]


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

full_df, mins, maxs = prepare(DATA_DIR)
train_df, val_df = split_by_engine(full_df)
X_tr,  y_rul_tr,  y_fore_tr  = make_windows(train_df)
X_val, y_rul_val, y_fore_val = make_windows(val_df)

BATCH_SIZE   = 256
train_loader = DataLoader(CMAPSSDataset(X_tr,  y_rul_tr,  y_fore_tr),  batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CMAPSSDataset(X_val, y_rul_val, y_fore_val), batch_size=BATCH_SIZE, shuffle=False)

print(f'train windows: {len(X_tr)}   val windows: {len(X_val)}')
print(f'X shape: {X_tr.shape}  y_fore shape: {y_fore_tr.shape}  y_rul shape: {y_rul_tr.shape}')

## Model: SentinelRUL

Same GRU backbone as Stage 6, now paired with both the forecast head and a new RUL head. The backbone
is initialised from the forecast checkpoint below, then fine tuned jointly with both heads.

In [ ]:
class GRUBackbone(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_dim, hidden_dim, num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.hidden_dim = hidden_dim

    def forward(self, x):
        out, _ = self.gru(x)
        return out  # (batch, seq, hidden_dim)


class ForecastHead(nn.Module):
    def __init__(self, hidden_dim, output_dim, horizon):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim * horizon),
        )
        self.output_dim = output_dim
        self.horizon    = horizon

    def forward(self, gru_out):
        last = gru_out[:, -1, :]
        return self.net(last).view(-1, self.horizon, self.output_dim)


class RULHead(nn.Module):
    def __init__(self, hidden_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, encoding):
        return self.net(encoding).squeeze(-1)


class SentinelRUL(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, n_layers=2, dropout=0.2, horizon=5):
        super().__init__()
        self.backbone      = GRUBackbone(input_dim, hidden_dim, n_layers, dropout)
        self.forecast_head = ForecastHead(hidden_dim, input_dim, horizon)
        self.rul_head      = RULHead(hidden_dim, dropout)

    def forward(self, x):
        gru_out  = self.backbone(x)
        encoding = gru_out[:, -1, :]
        forecast = self.forecast_head(gru_out)
        rul      = self.rul_head(encoding)
        return forecast, rul


class JointLoss(nn.Module):
    def __init__(self, forecast_weight=0.5, rul_weight=1.0):
        super().__init__()
        self.forecast_weight = forecast_weight
        self.rul_weight      = rul_weight
        self.mse = nn.MSELoss()

    def forward(self, forecast_pred, rul_pred, y_fore, y_rul):
        forecast_loss = self.mse(forecast_pred, y_fore)
        rul_loss      = self.mse(rul_pred, y_rul)
        total = self.forecast_weight * forecast_loss + self.rul_weight * rul_loss
        return total, {'forecast': forecast_loss.detach(), 'rul': rul_loss.detach()}


INPUT_DIM  = len(SENSOR_COLS)
HIDDEN_DIM = 128
N_LAYERS   = 2
DROPOUT    = 0.2
HORIZON    = FORECAST_HORIZON

model = SentinelRUL(INPUT_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, HORIZON).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {total_params:,}')
print(model)